[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/07_linear_least_squares/exercises.ipynb)

# Exercises — Topic 07: Linear Least Squares

20 fully solved problems in 4 levels: concept checks, foundational computations and derivations, AI/ML and physics applications, and challenge proofs.

## Level 0 — Concept Check

### Problem L0.1: Why does an overdetermined system have no solution?

Explain geometrically why $A\mathbf{x} = \mathbf{b}$ with $A \in \mathbb{R}^{m \times n}$, $m \gt n$, is generically unsolvable, and state exactly when it *is* solvable. Illustrate with $A = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1\end{bmatrix}$ and two different right-hand sides.

**Solution.**

The set of vectors reachable by $A$ is its column space $\mathcal{R}(A) = \{A\mathbf{x} : \mathbf{x} \in \mathbb{R}^{n}\}$, a subspace of $\mathbb{R}^{m}$ of dimension $r = \operatorname{rank}(A) \le n \lt m$. A subspace of dimension $r \lt m$ has **measure zero** in $\mathbb{R}^{m}$, so a $\mathbf{b}$ produced by noisy measurement lies in it with probability zero.

$$
A\mathbf{x} = \mathbf{b} \text{ is solvable} \iff \mathbf{b} \in \mathcal{R}(A) .
$$

For the given $A$, $\mathcal{R}(A)$ is the plane $\{(x_1, x_2, x_1 + x_2)\}$ in $\mathbb{R}^{3}$, i.e. the plane $b_3 = b_1 + b_2$.

- $\mathbf{b} = (1, 2, 3)^{\top}$: satisfies $3 = 1 + 2$, so it lies in the plane and $\mathbf{x} = (1,2)^{\top}$ solves the system exactly (residual zero).
- $\mathbf{b} = (1, 2, 4)^{\top}$: $4 \neq 3$, so $\mathbf{b}$ is off the plane by the vector $(0,0,1)^{\top}$; **no** $\mathbf{x}$ works, and least squares returns the projection instead.

$$
\boxed{A\mathbf{x} = \mathbf{b} \text{ solvable} \iff \mathbf{b} \in \mathcal{R}(A); \text{ otherwise minimize } \Vert A\mathbf{x} - \mathbf{b} \Vert_2}
$$

*Key takeaway:* Overdetermination is not a defect of the data — it is redundancy, and least squares converts redundancy into noise averaging by projecting $\mathbf{b}$ onto the reachable plane.

### Problem L0.2: Testing a candidate solution with orthogonality

For $A = \begin{bmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3\end{bmatrix}$ and $\mathbf{b} = (2, 3, 5)^{\top}$, decide which of $\mathbf{x}_1 = (0, 1)^{\top}$ and $\mathbf{x}_2 = (1/3, 3/2)^{\top}$ is the least-squares solution — using only the orthogonality condition, without solving anything.

**Solution.**

The optimality test is $A^{\top}\mathbf{r} = \mathbf{0}$ with $\mathbf{r} = \mathbf{b} - A\mathbf{x}$; the first row of $A^{\top}$ is all ones, so it asks that the residuals *sum to zero*, and the second row asks that $\sum_i t_i r_i = 0$ with $t = (1,2,3)$.

**Candidate $\mathbf{x}_1 = (0,1)^{\top}$:** $A\mathbf{x}_1 = (1, 2, 3)^{\top}$, so $\mathbf{r} = (1, 1, 2)^{\top}$ and

$$
A^{\top}\mathbf{r} = \begin{pmatrix} 1 + 1 + 2 \\ 1 + 2 + 6 \end{pmatrix} = \begin{pmatrix} 4 \\ 9 \end{pmatrix} \neq \mathbf{0} .
$$

Not optimal — the residual still has a component inside $\mathcal{R}(A)$ that can be removed.

**Candidate $\mathbf{x}_2 = (1/3, 3/2)^{\top}$:** $A\mathbf{x}_2 = (11/6, 10/3, 29/6)^{\top}$, so $\mathbf{r} = (1/6, -1/3, 1/6)^{\top}$ and

$$
A^{\top}\mathbf{r} = \begin{pmatrix} \tfrac{1}{6} - \tfrac{1}{3} + \tfrac{1}{6} \\ \tfrac{1}{6} - \tfrac{2}{3} + \tfrac{1}{2} \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \end{pmatrix} .
$$

Optimal. Note $\Vert \mathbf{r}_1 \Vert_2^2 = 6$ versus $\Vert \mathbf{r}_2 \Vert_2^2 = 1/6$, confirming the verdict.

$$
\boxed{\mathbf{x}_2 = (1/3,\ 3/2)^{\top} \text{ is the least-squares solution, since } A^{\top}\mathbf{r} = \mathbf{0}}
$$

*Key takeaway:* Optimality in least squares is a *checkable local condition* — orthogonality of the residual to every column — not something requiring a comparison against all other candidates.

### Problem L0.3: How many digits do the normal equations cost?

A design matrix has $\kappa_2(A) = 10^{5}$ and the computation runs in IEEE double precision ($\varepsilon_{\text{mach}} \approx 1.1 \times 10^{-16}$). Estimate the relative error of the solution computed (a) by Householder QR on a well-fitting problem, (b) by Cholesky on the normal equations. What happens at $\kappa_2(A) = 10^{9}$?

**Solution.**

A backward-stable solve of a system with condition number $\kappa$ returns a relative error of about $\varepsilon_{\text{mach}}\,\kappa$.

**(a) Householder QR.** It works with $A$ itself, so for a problem with a small residual ($\tan\theta \approx 0$) the error is

$$
\frac{\Vert \tilde{\mathbf{x}} - \hat{\mathbf{x}} \Vert_2}{\Vert \hat{\mathbf{x}} \Vert_2} \approx \varepsilon_{\text{mach}}\,\kappa_2(A) = 1.1 \times 10^{-16} \cdot 10^{5} \approx 10^{-11},
$$

about 11 correct digits.

**(b) Normal equations.** Cholesky operates on $A^{\top}A$, whose condition number is $\kappa_2(A)^2 = 10^{10}$:

$$
\approx 1.1 \times 10^{-16} \cdot 10^{10} \approx 10^{-6},
$$

about 6 correct digits — **five digits thrown away** by the choice of algorithm alone.

**At $\kappa_2(A) = 10^{9}$:** QR gives $\approx 10^{-7}$ (still usable); the normal equations give $\varepsilon_{\text{mach}}\cdot 10^{18} \approx 10^{2}$ — no correct digits, and Cholesky will typically abort because the rounded $A^{\top}A$ fails the positive-definiteness test.

$$
\boxed{\text{QR: } \sim \varepsilon \kappa = 10^{-11}; \quad \text{normal equations: } \sim \varepsilon \kappa^2 = 10^{-6}; \quad \text{rule: } \kappa \gt \varepsilon^{-1/2} \approx 10^{8} \Rightarrow \text{normal equations fail}}
$$

*Key takeaway:* The break-even point is $\kappa_2(A) \approx \varepsilon_{\text{mach}}^{-1/2} \approx 10^{8}$: beyond it the normal equations return garbage while QR still works.

### Problem L0.4: Choosing a method

Pick the appropriate algorithm for each: (a) $m = 10^{6}$ rows, $n = 5$ well-scaled columns, throughput critical; (b) a degree-12 polynomial fit on $[0, 1]$; (c) a design matrix with two exactly duplicated feature columns; (d) an image-deblurring problem whose singular values decay smoothly to $10^{-14}$.

**Solution.**

**(a) Normal equations with Cholesky.** With $n = 5$, $\kappa_2(A)$ is small and well-scaled, so squaring it is harmless; cost $mn^2 + \tfrac{1}{3}n^3$ versus $2mn^2$ for QR means a genuine $2\times$ saving, and the $A^{\top}A$ accumulation streams over the rows in one pass — ideal for out-of-core or online data.

**(b) QR (Householder), after changing basis.** The monomial Vandermonde matrix at degree 12 has $\kappa_2 \sim 10^{9}$; normal equations are hopeless. But the real fix is the *basis*: map $[0,1] \to [-1,1]$ and fit in Chebyshev polynomials, which drops $\kappa_2$ by many orders of magnitude. Then QR is comfortable.

**(c) SVD, or QR with column pivoting.** Duplicated columns make $\operatorname{rank}(A) \lt n$ exactly: $A^{\top}A$ is singular and Cholesky fails outright. The SVD detects the numerical rank and returns the minimum-norm solution $A^{+}\mathbf{b}$; column-pivoted QR is the cheaper alternative and additionally identifies which column to drop.

**(d) Regularization (Tikhonov or truncated SVD), computed via SVD.** With $\sigma_i \to 10^{-14}$ the problem is discretely ill-posed: the unregularized solution amplifies noise by $1/\sigma_i \approx 10^{14}$ and is pure garbage. Compute the SVD, choose $\lambda$ (discrepancy principle, GCV, or L-curve corner), and apply the filter factors $\sigma_i^2/(\sigma_i^2 + \lambda)$.

$$
\boxed{\text{(a) normal equations, (b) QR in a Chebyshev basis, (c) SVD / pivoted QR, (d) regularized SVD}}
$$

*Key takeaway:* The decision variables are $\kappa_2(A)$, whether the rank is certain, and whether the problem is ill-posed — not the size of $m$ alone.

## Level 1 — Foundation

### Problem L1.1: A line fit by the normal equations

Fit $y = c_0 + c_1 t$ to the data $(1, 2)$, $(2, 3)$, $(3, 5)$ by solving the normal equations by hand. Report the fitted coefficients, the residual vector, and the residual norm.

**Solution.**

Set up the design matrix and right-hand side:

$$
A = \begin{bmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \end{bmatrix}, \qquad \mathbf{b} = \begin{pmatrix} 2 \\ 3 \\ 5 \end{pmatrix} .
$$

Form the (tiny, so safe here) cross products:

$$
A^{\top}A = \begin{bmatrix} 3 & 6 \\ 6 & 14 \end{bmatrix}, \qquad A^{\top}\mathbf{b} = \begin{pmatrix} 2+3+5 \\ 2 + 6 + 15 \end{pmatrix} = \begin{pmatrix} 10 \\ 23 \end{pmatrix} .
$$

The determinant is $3 \cdot 14 - 36 = 6$, so

$$
\hat{\mathbf{c}} = \frac{1}{6}\begin{bmatrix} 14 & -6 \\ -6 & 3 \end{bmatrix}\begin{pmatrix} 10 \\ 23 \end{pmatrix} = \frac{1}{6}\begin{pmatrix} 140 - 138 \\ -60 + 69 \end{pmatrix} = \frac{1}{6}\begin{pmatrix} 2 \\ 9 \end{pmatrix} = \begin{pmatrix} 1/3 \\ 3/2 \end{pmatrix} .
$$

Fitted values $A\hat{\mathbf{c}} = (11/6,\ 10/3,\ 29/6)^{\top}$, so

$$
\mathbf{r} = \mathbf{b} - A\hat{\mathbf{c}} = \left(\tfrac{1}{6},\ -\tfrac{1}{3},\ \tfrac{1}{6}\right)^{\top}, \qquad \Vert \mathbf{r} \Vert_2^2 = \tfrac{1}{36} + \tfrac{4}{36} + \tfrac{1}{36} = \tfrac{1}{6} .
$$

$$
\boxed{y = \tfrac{1}{3} + \tfrac{3}{2}t, \qquad \mathbf{r} = \left(\tfrac{1}{6}, -\tfrac{1}{3}, \tfrac{1}{6}\right)^{\top}, \qquad \Vert \mathbf{r} \Vert_2 = 1/\sqrt{6} \approx 0.4082}
$$

*Key takeaway:* The normal equations reduce an $m \times n$ fitting problem to an $n \times n$ symmetric positive definite system — here a $2 \times 2$ solve — which is why they remain the standard *derivation*, whatever algorithm actually computes the answer.

### Problem L1.2: The same fit by Gram–Schmidt QR

Compute a thin QR factorization of the $A$ of Problem L1.1 by (modified) Gram–Schmidt, solve $R\hat{\mathbf{c}} = Q^{\top}\mathbf{b}$ by back substitution, and recover $\Vert \mathbf{r} \Vert_2$ from $\Vert \mathbf{b} \Vert_2$ and $\Vert Q^{\top}\mathbf{b} \Vert_2$ without computing $\mathbf{r}$.

**Solution.**

**Step 1 — first column.** $\mathbf{a}_1 = (1,1,1)^{\top}$, $r_{11} = \Vert \mathbf{a}_1 \Vert_2 = \sqrt{3}$, so $\mathbf{q}_1 = \tfrac{1}{\sqrt{3}}(1,1,1)^{\top}$.

**Step 2 — second column.** $\mathbf{a}_2 = (1,2,3)^{\top}$, $r_{12} = \mathbf{q}_1^{\top}\mathbf{a}_2 = \tfrac{6}{\sqrt{3}} = 2\sqrt{3}$, and

$$
\mathbf{v} = \mathbf{a}_2 - r_{12}\mathbf{q}_1 = (1,2,3)^{\top} - 2(1,1,1)^{\top} = (-1, 0, 1)^{\top}, \qquad r_{22} = \Vert \mathbf{v} \Vert_2 = \sqrt{2},
$$

so $\mathbf{q}_2 = \tfrac{1}{\sqrt{2}}(-1, 0, 1)^{\top}$. Hence

$$
Q = \begin{bmatrix} 1/\sqrt{3} & -1/\sqrt{2} \\ 1/\sqrt{3} & 0 \\ 1/\sqrt{3} & 1/\sqrt{2} \end{bmatrix}, \qquad R = \begin{bmatrix} \sqrt{3} & 2\sqrt{3} \\ 0 & \sqrt{2} \end{bmatrix} .
$$

**Step 3 — solve.** $Q^{\top}\mathbf{b} = \bigl(\tfrac{2+3+5}{\sqrt{3}},\ \tfrac{-2+5}{\sqrt{2}}\bigr)^{\top} = \bigl(\tfrac{10}{\sqrt{3}},\ \tfrac{3}{\sqrt{2}}\bigr)^{\top}$. Back substitution:

$$
c_1 = \frac{3/\sqrt{2}}{\sqrt{2}} = \frac{3}{2}, \qquad c_0 = \frac{10/\sqrt{3} - 2\sqrt{3}\cdot \tfrac{3}{2}}{\sqrt{3}} = \frac{10}{3} - 3 = \frac{1}{3}.
$$

Identical to L1.1, as it must be in exact arithmetic.

**Step 4 — residual for free.** Since $\Vert \mathbf{b} \Vert_2^2 = \Vert Q^{\top}\mathbf{b} \Vert_2^2 + \Vert Q_\perp^{\top}\mathbf{b} \Vert_2^2$,

$$
\Vert \mathbf{r} \Vert_2^2 = \Vert \mathbf{b} \Vert_2^2 - \Vert Q^{\top}\mathbf{b} \Vert_2^2 = (4 + 9 + 25) - \left(\frac{100}{3} + \frac{9}{2}\right) = 38 - \frac{227}{6} = \frac{1}{6} .
$$

$$
\boxed{R = \begin{bmatrix} \sqrt{3} & 2\sqrt{3} \\ 0 & \sqrt{2}\end{bmatrix}, \quad \hat{\mathbf{c}} = (1/3,\ 3/2)^{\top}, \quad \Vert \mathbf{r} \Vert_2^2 = \Vert \mathbf{b} \Vert_2^2 - \Vert Q^{\top}\mathbf{b} \Vert_2^2 = 1/6}
$$

*Key takeaway:* QR gives the same answer as the normal equations in exact arithmetic — and the residual norm falls out of the Pythagorean identity, with $R$ being precisely the Cholesky factor of $A^{\top}A$ (indeed $R^{\top}R = \begin{bmatrix} 3 & 6 \\ 6 & 14\end{bmatrix}$).

### Problem L1.3: Building a Householder reflector

Construct the Householder reflector $H$ that maps $\mathbf{a} = (1, 1, 1)^{\top}$ to a multiple of $\mathbf{e}_1$. Verify $H$ is orthogonal and symmetric, compute $H\mathbf{a}$, and explain why the sign convention $\alpha = -\operatorname{sign}(a_1)\Vert \mathbf{a}\Vert_2$ matters.

**Solution.**

With $\Vert \mathbf{a} \Vert_2 = \sqrt{3}$ and $a_1 = 1 \gt 0$, the stable sign choice is $\alpha = -\sqrt{3}$. Then

$$
\mathbf{v} = \mathbf{a} - \alpha\mathbf{e}_1 = (1 + \sqrt{3},\ 1,\ 1)^{\top}, \qquad \mathbf{v}^{\top}\mathbf{v} = (1+\sqrt{3})^2 + 2 = 6 + 2\sqrt{3} \approx 9.4641 .
$$

The reflector is

$$
H = I - \frac{2\,\mathbf{v}\mathbf{v}^{\top}}{\mathbf{v}^{\top}\mathbf{v}} \approx \begin{bmatrix} -0.5774 & -0.5774 & -0.5774 \\ -0.5774 & 0.7887 & -0.2113 \\ -0.5774 & -0.2113 & 0.7887 \end{bmatrix}.
$$

**Symmetry** is immediate ($\mathbf{v}\mathbf{v}^{\top}$ is symmetric). **Orthogonality**: writing $\beta = 2/(\mathbf{v}^{\top}\mathbf{v})$,

$$
H^{\top}H = H^2 = I - 2\beta\mathbf{v}\mathbf{v}^{\top} + \beta^2 \mathbf{v}(\mathbf{v}^{\top}\mathbf{v})\mathbf{v}^{\top} = I - 2\beta \mathbf{v}\mathbf{v}^{\top} + 2\beta\mathbf{v}\mathbf{v}^{\top} = I .
$$

**Action:** $\mathbf{v}^{\top}\mathbf{a} = (1+\sqrt{3}) + 1 + 1 = 3 + \sqrt{3}$, and $\beta \mathbf{v}^{\top}\mathbf{a} = \frac{2(3+\sqrt{3})}{6 + 2\sqrt{3}} = 1$, so

$$
H\mathbf{a} = \mathbf{a} - \mathbf{v} = (1,1,1)^{\top} - (1+\sqrt{3}, 1, 1)^{\top} = (-\sqrt{3}, 0, 0)^{\top} = \alpha \mathbf{e}_1 .
$$

**The sign.** Choosing $\alpha = +\Vert \mathbf{a}\Vert_2$ instead would give $v_1 = a_1 - \Vert \mathbf{a} \Vert_2$, a subtraction of nearly equal numbers whenever $\mathbf{a}$ is already close to $\mathbf{e}_1$ — catastrophic cancellation that destroys the accuracy of $\mathbf{v}$ and hence of $Q$. The convention $\alpha = -\operatorname{sign}(a_1)\Vert \mathbf{a}\Vert_2$ makes $\vert v_1 \vert = \vert a_1 \vert + \Vert \mathbf{a}\Vert_2$ an addition of like signs.

$$
\boxed{\mathbf{v} = (1+\sqrt{3},\,1,\,1)^{\top}, \qquad H\mathbf{a} = (-\sqrt{3},\,0,\,0)^{\top}}
$$

*Key takeaway:* One reflector annihilates an entire column below the diagonal, is applied in $O(m)$ flops as a rank-one update, and never needs to be formed explicitly — that combination is what makes Householder QR both fast and backward stable.

### Problem L1.4: A rank-deficient problem and the minimum-norm solution

For $A = \begin{bmatrix} 1 & 2 \\ 2 & 4\end{bmatrix}$ and $\mathbf{b} = (3, 6)^{\top}$, describe the full solution set of the least-squares problem and identify the minimum-norm solution via the SVD.

**Solution.**

**Rank and consistency.** The second row is twice the first, so $\operatorname{rank}(A) = 1$ and $\mathcal{R}(A) = \operatorname{span}\{(1,2)^{\top}\}$. Since $\mathbf{b} = 3\,(1,2)^{\top}$ lies in that span, the residual is *zero*: this is a consistent but underdetermined-in-disguise problem.

**Solution set.** $A\mathbf{x} = \mathbf{b}$ reduces to the single equation $x_1 + 2x_2 = 3$. The null space is $\mathcal{N}(A) = \operatorname{span}\{(-2, 1)^{\top}\}$, so

$$
\text{solution set} = \{(3, 0)^{\top} + t\,(-2, 1)^{\top} : t \in \mathbb{R}\},
$$

a line — consistent with $\dim = n - r = 2 - 1 = 1$.

**SVD.** $A = \sigma_1 \mathbf{u}_1\mathbf{v}_1^{\top}$ with $\sigma_1 = \Vert A \Vert_F = \sqrt{1+4+4+16} = 5$, $\mathbf{u}_1 = \tfrac{1}{\sqrt{5}}(1,2)^{\top}$, $\mathbf{v}_1 = \tfrac{1}{\sqrt{5}}(1,2)^{\top}$, and $\sigma_2 = 0$. Then $\mathbf{u}_1^{\top}\mathbf{b} = \tfrac{3 + 12}{\sqrt{5}} = \tfrac{15}{\sqrt{5}} = 3\sqrt{5}$, so

$$
\hat{\mathbf{x}}_{\min} = A^{+}\mathbf{b} = \frac{\mathbf{u}_1^{\top}\mathbf{b}}{\sigma_1}\mathbf{v}_1 = \frac{3\sqrt{5}}{5}\cdot\frac{1}{\sqrt{5}}\begin{pmatrix}1\\2\end{pmatrix} = \begin{pmatrix} 3/5 \\ 6/5 \end{pmatrix} .
$$

**Check of minimality.** On the line, $\Vert \mathbf{x}(t) \Vert_2^2 = (3-2t)^2 + t^2 = 5t^2 - 12t + 9$, minimized at $t = 6/5$, giving $\mathbf{x} = (3 - 12/5,\ 6/5)^{\top} = (3/5, 6/5)^{\top}$ with $\Vert \mathbf{x} \Vert_2 = 3/\sqrt{5} \approx 1.3416$ — matching $A^{+}\mathbf{b}$, and note it is orthogonal to $(-2,1)^{\top}$, i.e. it lies in $\mathcal{N}(A)^{\perp}$ as Proof 5 predicts.

$$
\boxed{\{(3,0)^{\top} + t(-2,1)^{\top}\}; \qquad \hat{\mathbf{x}}_{\min} = A^{+}\mathbf{b} = (3/5,\ 6/5)^{\top}, \quad \Vert \hat{\mathbf{x}}_{\min} \Vert_2 = 3/\sqrt{5}}
$$

*Key takeaway:* Rank deficiency destroys uniqueness, not existence; the pseudoinverse restores uniqueness by the extra rule "smallest norm", which geometrically means dropping every null-space component.

### Problem L1.5: Ridge regression on the line-fit data

Solve the ridge problem for the data of Problem L1.1 with $\lambda = 1$, compare the coefficients and residual with the unregularized fit, and state the augmented-matrix formulation.

**Solution.**

**Augmented form.** The ridge problem $\min_{\mathbf{c}} \Vert A\mathbf{c} - \mathbf{b} \Vert_2^2 + \lambda\Vert \mathbf{c} \Vert_2^2$ is the ordinary least-squares problem for

$$
A_\lambda = \begin{bmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \\ 1 & 0 \\ 0 & 1 \end{bmatrix}, \qquad \mathbf{b}_\lambda = (2, 3, 5, 0, 0)^{\top} \qquad (\sqrt{\lambda} = 1),
$$

so it can be solved by plain QR with no cross products at all.

**Normal equations of the augmented problem.** $(A^{\top}A + I)\hat{\mathbf{c}}_\lambda = A^{\top}\mathbf{b}$:

$$
\begin{bmatrix} 4 & 6 \\ 6 & 15 \end{bmatrix}\hat{\mathbf{c}}_\lambda = \begin{pmatrix} 10 \\ 23 \end{pmatrix}, \qquad \det = 60 - 36 = 24 ,
$$

$$
\hat{\mathbf{c}}_\lambda = \frac{1}{24}\begin{bmatrix} 15 & -6 \\ -6 & 4 \end{bmatrix}\begin{pmatrix} 10 \\ 23 \end{pmatrix} = \frac{1}{24}\begin{pmatrix} 150 - 138 \\ -60 + 92 \end{pmatrix} = \frac{1}{24}\begin{pmatrix} 12 \\ 32 \end{pmatrix} = \begin{pmatrix} 1/2 \\ 4/3 \end{pmatrix} .
$$

**Comparison.** Unregularized: $(1/3, 3/2)$ with $\Vert \hat{\mathbf{c}} \Vert_2^2 = \tfrac{1}{9} + \tfrac{9}{4} = \tfrac{85}{36} \approx 2.361$ and $\Vert \mathbf{r} \Vert_2^2 = 1/6 \approx 0.1667$. Ridge: $(1/2, 4/3)$ with $\Vert \hat{\mathbf{c}}_\lambda \Vert_2^2 = \tfrac{1}{4} + \tfrac{16}{9} = \tfrac{73}{36} \approx 2.028$ and fitted values $(11/6, 19/6, 9/2)$, giving $\mathbf{r}_\lambda = (1/6, -1/6, 1/2)^{\top}$, $\Vert \mathbf{r}_\lambda \Vert_2^2 = \tfrac{1+1+9}{36} = \tfrac{11}{36} \approx 0.3056$.

The slope shrank from $1.5$ to $1.333$ and the coefficient norm dropped, at the cost of a larger residual — the bias–variance trade in miniature. Note also $\kappa_2(A^{\top}A) \approx 46.1$, whereas $A^{\top}A + I$ has eigenvalues $\tfrac{19 \pm \sqrt{265}}{2} \approx 17.64$ and $1.36$, so $\kappa_2(A^{\top}A + I) \approx 13.0$: better conditioned as well as smaller.

$$
\boxed{\hat{\mathbf{c}}_{\lambda=1} = (1/2,\ 4/3)^{\top}; \quad \Vert \hat{\mathbf{c}}_\lambda \Vert_2 \lt \Vert \hat{\mathbf{c}} \Vert_2, \quad \Vert \mathbf{r}_\lambda \Vert_2 \gt \Vert \mathbf{r} \Vert_2}
$$

*Key takeaway:* Ridge always increases the residual and decreases the coefficient norm — both monotonically in $\lambda$ — and its augmented form means it needs no new algorithm, just a taller matrix.

### Problem L1.6: The projector and the residual as complementary projections

Using the $Q$ from Problem L1.2, form $P = QQ^{\top}$. Verify $P^2 = P$, $P^{\top} = P$, $\operatorname{tr}(P) = 2$, and show that $\mathbf{r} = (I - P)\mathbf{b}$.

**Solution.**

With $\mathbf{q}_1 = \tfrac{1}{\sqrt{3}}(1,1,1)^{\top}$ and $\mathbf{q}_2 = \tfrac{1}{\sqrt{2}}(-1,0,1)^{\top}$,

$$
P = \mathbf{q}_1\mathbf{q}_1^{\top} + \mathbf{q}_2\mathbf{q}_2^{\top} = \frac{1}{3}\begin{bmatrix} 1&1&1\\1&1&1\\1&1&1\end{bmatrix} + \frac{1}{2}\begin{bmatrix} 1&0&-1\\0&0&0\\-1&0&1\end{bmatrix} = \frac{1}{6}\begin{bmatrix} 5&2&-1\\2&2&2\\-1&2&5\end{bmatrix} .
$$

**Symmetry** is visible. **Idempotence**: $P^2 = QQ^{\top}QQ^{\top} = Q(Q^{\top}Q)Q^{\top} = QI_2Q^{\top} = P$, since the columns of $Q$ are orthonormal. **Trace**: $\operatorname{tr}(P) = \tfrac{5 + 2 + 5}{6} = 2 = \operatorname{rank}(A)$ — for an orthogonal projector the trace equals the dimension of the range, because its eigenvalues are $1$ ($\times 2$) and $0$ ($\times 1$).

**Residual.** $I - P = \tfrac{1}{6}\begin{bmatrix} 1&-2&1\\-2&4&-2\\1&-2&1\end{bmatrix} = \tfrac{1}{6}\mathbf{w}\mathbf{w}^{\top}$ with $\mathbf{w} = (1,-2,1)^{\top}$ — a rank-one projector onto $\mathcal{R}(A)^{\perp} = \mathcal{N}(A^{\top})$, and indeed $A^{\top}\mathbf{w} = (1-2+1,\ 1-4+3)^{\top} = \mathbf{0}$. Applying it,

$$
(I - P)\mathbf{b} = \frac{\mathbf{w}^{\top}\mathbf{b}}{6}\mathbf{w} = \frac{2 - 6 + 5}{6}(1,-2,1)^{\top} = \left(\tfrac{1}{6}, -\tfrac{1}{3}, \tfrac{1}{6}\right)^{\top} = \mathbf{r} .
$$

$$
\boxed{P = \tfrac{1}{6}\begin{bmatrix} 5&2&-1\\2&2&2\\-1&2&5\end{bmatrix}, \quad P^2 = P = P^{\top}, \quad \operatorname{tr}P = 2, \quad \mathbf{r} = (I-P)\mathbf{b}}
$$

*Key takeaway:* Least squares splits $\mathbf{b} = P\mathbf{b} + (I-P)\mathbf{b}$ into "explained" and "unexplained" orthogonal parts; $\operatorname{tr}(P)$ is the model's degrees of freedom, which is exactly the quantity ridge regression fractionalizes.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: The Läuchli matrix — when the normal equations destroy the data

Let $\varepsilon = 10^{-8}$ and

$$
A = \begin{bmatrix} 1 & 1 \\ \varepsilon & 0 \\ 0 & \varepsilon \end{bmatrix} .
$$

Compute $\kappa_2(A)$ exactly, form $A^{\top}A$ exactly and in IEEE double precision, and explain the consequence for a solve.

**Solution.**

**Exact singular values.** $A^{\top}A = \begin{bmatrix} 1+\varepsilon^2 & 1 \\ 1 & 1 + \varepsilon^2\end{bmatrix}$ has eigenvalues $(1+\varepsilon^2) \pm 1$, i.e. $2 + \varepsilon^2$ and $\varepsilon^2$. Therefore

$$
\sigma_1 = \sqrt{2 + \varepsilon^2} \approx \sqrt{2}, \qquad \sigma_2 = \varepsilon, \qquad \kappa_2(A) = \frac{\sqrt{2 + \varepsilon^2}}{\varepsilon} \approx \frac{\sqrt{2}}{10^{-8}} \approx 1.414 \times 10^{8} .
$$

$A$ has full column rank 2, comfortably; $\sigma_2 = 10^{-8}$ is fifteen thousand times larger than $\varepsilon_{\text{mach}}$.

**Floating-point cross product.** $\varepsilon^2 = 10^{-16} \lt \varepsilon_{\text{mach}} \approx 1.1 \times 10^{-16}$, so $\mathrm{fl}(1 + \varepsilon^2) = 1$ exactly, and the *computed* matrix is

$$
\mathrm{fl}(A^{\top}A) = \begin{bmatrix} 1 & 1 \\ 1 & 1 \end{bmatrix}, \qquad \det = 0 .
$$

A numerically **singular** matrix. Cholesky fails (the second pivot is $0$ or negative), and any solve returns a meaningless answer or an error. Consistently, $\kappa_2(A^{\top}A) = \kappa_2(A)^2 = 2 \times 10^{16}$ and $\varepsilon_{\text{mach}}\kappa_2(A)^2 \approx 2.2 \gt 1$: the error bound exceeds the answer.

**What QR does instead.** Householder QR applies orthogonal transformations to $A$ directly. Nothing of size $\varepsilon^2$ is ever added to something of size $1$, the two columns stay distinguishable, and the computed solution has relative error $\approx \varepsilon_{\text{mach}}\kappa_2(A) \approx 10^{-8}$ — eight correct digits where the normal equations produced none.

$$
\boxed{\kappa_2(A) = \tfrac{\sqrt{2+\varepsilon^2}}{\varepsilon} \approx 1.41 \times 10^{8}; \quad \mathrm{fl}(A^{\top}A) = \begin{bmatrix} 1&1\\1&1\end{bmatrix} \text{ is singular, while } A \text{ has rank } 2}
$$

*Key takeaway:* The information is destroyed by the *multiplication*, before any solve begins — no downstream cleverness can recover it. This single $3 \times 2$ example is the standard argument for QR over the normal equations.

### Problem L2.2: Ridge filter factors and noise amplification

A design matrix has singular values $\sigma = (10,\ 1,\ 0.01)$. Compute the ridge filter factors for $\lambda = 10^{-2}$, the noise amplification in each direction with and without regularization, and the effective degrees of freedom.

**Solution.**

**Filter factors** $f_i = \sigma_i^2/(\sigma_i^2 + \lambda)$ with $\lambda = 0.01$:

| $\sigma_i$ | $\sigma_i^2$ | $f_i = \sigma_i^2/(\sigma_i^2 + \lambda)$ | unregularized gain $1/\sigma_i$ | ridge gain $\sigma_i/(\sigma_i^2+\lambda)$ |
| :--- | :--- | :--- | :--- | :--- |
| $10$ | $100$ | $0.99990$ | $0.1$ | $0.09999$ |
| $1$ | $1$ | $0.99010$ | $1$ | $0.99010$ |
| $0.01$ | $10^{-4}$ | $0.00990$ | $100$ | $0.99010$ |

**Reading the table.** The two well-determined directions pass essentially untouched ($f \approx 1$). The third direction, which unregularized amplifies any noise component $\mathbf{u}_3^{\top}\boldsymbol{\varepsilon}$ by a factor $1/\sigma_3 = 100$, is damped by $f_3 \approx 0.0099$ so that its total gain is only $\sigma_3/(\sigma_3^2 + \lambda) = 0.99$ — a **hundredfold** reduction in noise amplification.

**Effective degrees of freedom.**

$$
\mathrm{df}(\lambda) = \operatorname{tr}\bigl[A(A^{\top}A + \lambda I)^{-1}A^{\top}\bigr] = \sum_i f_i = 0.99990 + 0.99010 + 0.00990 = 1.99990 .
$$

The model behaves like a 2-parameter model even though it has 3 columns: ridge has effectively deleted the third direction, which is precisely why it fixes multicollinearity. The general rule of thumb is that $\lambda$ acts as a soft threshold at $\sigma = \sqrt{\lambda} = 0.1$: directions well above it survive, those well below are removed.

$$
\boxed{f = (0.9999,\ 0.9901,\ 0.0099); \quad \text{noise gain } 100 \to 0.99; \quad \mathrm{df}(\lambda) = 1.9999}
$$

*Key takeaway:* Regularization is best understood in the SVD basis: it is a smooth low-pass filter on singular directions, with cutoff $\sqrt{\lambda}$, and its effective dimension $\sum_i f_i$ interpolates continuously between $n$ (at $\lambda = 0$) and $0$.

### Problem L2.3: Why high-degree polynomial fits lose their coefficients

Fitting $p(t) = \sum_{j=0}^{d} x_j t^j$ to $50$ equispaced points on $[0,1]$ gives the Vandermonde matrix $A_{ij} = t_i^{\,j}$. Its 2-norm condition numbers are approximately $6.4\times 10^{2}$ for $d = 4$, $3.6 \times 10^{6}$ for $d = 9$, and $2.3 \times 10^{10}$ for $d = 14$. Explain the growth, predict the accuracy of the fitted coefficients at each degree, and give the fix.

**Solution.**

**Why $\kappa$ explodes.** The columns are the sampled functions $1, t, t^2, \ldots, t^{d}$. On $[0,1]$ these become nearly indistinguishable as $j$ grows — $t^{13}$ and $t^{14}$ are both tiny except near $t=1$, where both are near $1$ — so the normalized inner product between consecutive columns tends to $1$. The columns are nearly parallel, which is exactly what a small $\sigma_{\min}$ means. Quantitatively the continuous analogue is the Hilbert matrix $H_{jk} = \int_0^1 t^{j+k}\,dt = 1/(j+k+1)$, whose condition number grows like $e^{3.5 d}$ — exponential in degree.

**Predicted accuracy** (Householder QR, $\varepsilon_{\text{mach}} \approx 1.1\times 10^{-16}$, relative error $\approx \varepsilon\kappa$):

| $d$ | $\kappa_2(A)$ | QR coefficient accuracy | normal-equations accuracy ($\varepsilon\kappa^2$) |
| :--- | :--- | :--- | :--- |
| $4$ | $6.4 \times 10^{2}$ | $\sim 7 \times 10^{-14}$ (13 digits) | $\sim 5\times 10^{-11}$ (10 digits) |
| $9$ | $3.6 \times 10^{6}$ | $\sim 4 \times 10^{-10}$ (9 digits) | $\sim 1.4\times 10^{-3}$ (2 digits) |
| $14$ | $2.3 \times 10^{10}$ | $\sim 2.5 \times 10^{-6}$ (5 digits) | $\gt 1$ (**no** digits) |

**The essential subtlety.** Even at $d = 14$ the *fitted curve* $p(t)$ may still be accurate to near machine precision, because the errors in the coefficients are strongly correlated and cancel when the polynomial is evaluated. What is destroyed is the *interpretability* of the individual $x_j$ — a reminder that a well-conditioned map ($\mathbf{x} \mapsto p$) can have a badly conditioned inverse.

**The fix: change basis.** Map $t \in [0,1]$ to $s = 2t - 1 \in [-1,1]$ and expand in Chebyshev polynomials $T_j(s)$. These are nearly orthogonal under the discrete inner product induced by well-chosen nodes, so the design matrix has $\kappa_2$ of order $10$ rather than $10^{10}$, and the coefficients become meaningful and individually interpretable. (Equivalently: use `numpy.polynomial.chebyshev` rather than raw `numpy.polyfit`, or fit splines — see Topic 04.)

$$
\boxed{\kappa_2(\text{Vandermonde}) \text{ grows exponentially in } d; \text{ at } d = 14 \text{ monomial coefficients carry } \approx 5 \text{ digits under QR and none under normal equations}}
$$

*Key takeaway:* Conditioning is a property of the *parameterization*, not of the underlying approximation problem — the same fit in an orthogonal basis is perfectly well conditioned.

### Problem L2.4: Weighted least squares for measurements with unequal error bars

Three independent laboratories measure the same physical constant $\mu$, reporting $b = (10.0,\ 14.0,\ 12.0)$ with standard deviations $\sigma = (1,\ 2,\ 1)$. Derive the weighted least-squares estimate, compare it with the unweighted mean, and give the uncertainty of each.

**Solution.**

**Model.** $A = (1,1,1)^{\top}$ (a single parameter $\mu$), $\mathbf{b}$ as given. Maximum likelihood for independent Gaussian errors maximizes $-\tfrac12\sum_i (b_i - \mu)^2/\sigma_i^2$, i.e. minimizes the weighted residual with $w_i = 1/\sigma_i^2$:

$$
\min_{\mu} \sum_{i=1}^{3} w_i (\mu - b_i)^2, \qquad w = \left(1,\ \tfrac14,\ 1\right).
$$

**Normal equation.** $A^{\top}WA\,\hat{\mu} = A^{\top}W\mathbf{b}$ becomes the scalar equation

$$
\hat{\mu} = \frac{\sum_i w_i b_i}{\sum_i w_i} = \frac{1(10) + \tfrac14(14) + 1(12)}{1 + \tfrac14 + 1} = \frac{10 + 3.5 + 12}{2.25} = \frac{25.5}{2.25} = \frac{34}{3} \approx 11.3333 .
$$

**Unweighted mean.** $\bar{b} = (10 + 14 + 12)/3 = 12$ — pulled upward by the *least reliable* measurement.

**Uncertainties.** For the weighted estimator, $\operatorname{Var}(\hat{\mu}) = 1/\sum_i w_i = 1/2.25 = 4/9$, so $\mathrm{sd} = 2/3 \approx 0.667$. For the plain mean, $\operatorname{Var}(\bar{b}) = \tfrac{1}{9}\sum_i \sigma_i^2 = \tfrac{1 + 4 + 1}{9} = \tfrac{6}{9}$, so $\mathrm{sd} = \sqrt{2/3} \approx 0.816$. Weighting reduces the variance by $33\%$; by Gauss–Markov, $\hat{\mu}$ is the minimum-variance unbiased *linear* estimator, so no other weighting can do better.

**Computationally**, do not form $A^{\top}WA$: scale row $i$ of $A$ and $\mathbf{b}$ by $\sqrt{w_i} = 1/\sigma_i$ and run ordinary QR on the whitened problem. For correlated errors with covariance $C = LL^{\top}$, solve in $L^{-1}A$, $L^{-1}\mathbf{b}$ instead.

$$
\boxed{\hat{\mu}_{\text{WLS}} = \tfrac{34}{3} \approx 11.333 \pm 0.667; \qquad \bar{b} = 12 \pm 0.816}
$$

*Key takeaway:* Weights are inverse variances, not arbitrary preferences; weighting is applied by scaling rows *before* factorization, and it is what makes least squares the maximum-likelihood estimator for heteroscedastic data.

### Problem L2.5: Closed form versus gradient descent for linear regression

A regression has $m = 10^{6}$ samples. Compare the cost of the closed-form solve with gradient descent for $n = 100$ and $n = 10^{5}$ features, and quantify how feature scaling (which reduces $\kappa_2(X)$ from $10^{3}$ to $10$) changes the iteration count for a $10^{-6}$ relative error.

**Solution.**

**Closed-form cost.** Normal equations plus Cholesky: $mn^2 + \tfrac13 n^3$ flops. Householder QR: $2mn^2 - \tfrac23 n^3$.

- $n = 100$: normal equations $\approx 1.0 \times 10^{10}$ flops, QR $\approx 2.0\times 10^{10}$ — seconds on a laptop. Memory for $X$ is $8\times 10^{8}$ bytes $= 0.8$ GB. **Feasible.**
- $n = 10^{5}$: $mn^2 = 10^{6}\cdot 10^{10} = 10^{16}$ flops, and $X$ alone needs $8\times 10^{11}$ bytes $= 800$ GB. **Infeasible** — the closed form is off the table.

**Gradient-descent cost.** Each step needs $X\mathbf{x}$ and $X^{\top}\mathbf{r}$: $\approx 4mn$ flops, i.e. $4\times 10^{8}$ for $n=100$ and $4\times 10^{11}$ for $n = 10^{5}$, and it never forms an $n \times n$ matrix. With optimal step size the error contracts by

$$
\rho = \frac{\kappa - 1}{\kappa + 1}, \qquad \kappa = \kappa_2(X^{\top}X) = \kappa_2(X)^2 ,
$$

so reaching relative error $\delta$ takes $N \approx \ln(1/\delta)/\ln(1/\rho)$ iterations.

| $\kappa_2(X)$ | $\kappa = \kappa_2(X)^2$ | $\rho$ | iterations for $\delta = 10^{-6}$ |
| :--- | :--- | :--- | :--- |
| $10^{3}$ (unscaled) | $10^{6}$ | $0.999998$ | $\approx 6.9 \times 10^{6}$ |
| $10^{2}$ | $10^{4}$ | $0.99980$ | $\approx 6.9 \times 10^{4}$ |
| $10$ (standardized) | $10^{2}$ | $0.9802$ | $\approx 691$ |

Standardizing the features cuts the iteration count by four orders of magnitude — from computationally hopeless to under a thousand steps. This is why `StandardScaler` before a gradient-based regressor is not cosmetic.

**Better iterative choice.** Conjugate gradients on the normal equations converge in $O(\sqrt{\kappa}) = O(\kappa_2(X))$ iterations, and **LSQR** achieves the same rate while applying $X$ and $X^{\top}$ only — never squaring the condition number numerically. That is the production choice for large sparse least squares.

$$
\boxed{n=100: \text{closed form}\ (10^{10}\text{ flops}); \quad n=10^{5}: \text{iterative}; \quad \kappa_2(X): 10^{3}\to 10 \Rightarrow 6.9\times10^{6} \to 691 \text{ steps}}
$$

*Key takeaway:* The closed form wins whenever $n$ is small enough that $O(mn^2)$ and $O(n^2)$ memory are affordable; beyond that, conditioning — and therefore preprocessing — becomes the dominant performance variable.

### Problem L2.6: Ridge as MAP estimation, and multicollinearity in practice

Show that ridge regression is the MAP estimate under a Gaussian prior, identify $\lambda$ in terms of the noise and prior variances, and explain the "exploding coefficients with alternating signs" symptom of collinear features.

**Solution.**

**MAP derivation.** Take the model $\mathbf{b} = A\mathbf{x} + \boldsymbol{\varepsilon}$ with $\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 I_m)$ and the prior $\mathbf{x} \sim \mathcal{N}(\mathbf{0}, \tau^2 I_n)$. Then

$$
-\log p(\mathbf{x} \mid \mathbf{b}) = \frac{1}{2\sigma^2}\Vert A\mathbf{x} - \mathbf{b} \Vert_2^2 + \frac{1}{2\tau^2}\Vert \mathbf{x} \Vert_2^2 + \text{const} .
$$

Multiplying by $2\sigma^2$ (which does not move the minimizer) gives exactly the ridge objective with

$$
\lambda = \frac{\sigma^2}{\tau^2} .
$$

Noisy data (large $\sigma$) or a confident prior (small $\tau$) both push $\lambda$ up. The posterior is Gaussian with covariance $\sigma^2 (A^{\top}A + \lambda I)^{-1}$, so ridge also comes with calibrated uncertainty — the Bayesian-linear-regression / GP-regression connection.

**The collinearity symptom.** Suppose two features are nearly identical, $\mathbf{a}_2 = \mathbf{a}_1 + \delta\mathbf{u}$ with $\delta$ tiny. Then $\mathbf{a}_1 - \mathbf{a}_2 \approx \mathbf{0}$, so the direction $\mathbf{v} \approx (1, -1, 0, \ldots)^{\top}/\sqrt{2}$ has $\Vert A\mathbf{v} \Vert_2 = \sigma_{\min} \approx \delta$: adding $t\mathbf{v}$ to any solution changes the fit by only $O(t\delta)$ while changing the coefficients by $O(t)$. The optimizer therefore has almost no incentive to control $t$, and noise in $\mathbf{b}$ sets it to a huge value of arbitrary sign — hence coefficients like $(+8.4\times10^{4}, -8.4\times 10^{4})$ on two nearly duplicate features, whose *sum* is stable and small.

**What ridge does.** The penalty $\lambda\Vert \mathbf{x} \Vert_2^2$ makes the objective strictly convex in the flat direction, since the Hessian becomes $2(A^{\top}A + \lambda I) \succeq 2\lambda I \succ 0$. The coefficient along $\mathbf{v}$ is damped by $f_{\min} = \sigma_{\min}^2/(\sigma_{\min}^2 + \lambda) \approx \delta^2/\lambda \approx 0$, so the pair receives an equal, split, and stable weight. (Ridge splits collinear features evenly; lasso instead picks one and zeros the other — the practical difference between $\ell_2$ and $\ell_1$ penalties.)

$$
\boxed{\text{ridge} = \text{MAP with } \mathbf{x} \sim \mathcal{N}(\mathbf{0}, \tau^2 I), \quad \lambda = \sigma^2/\tau^2; \quad \text{collinearity} \Rightarrow \sigma_{\min} \approx 0 \Rightarrow \text{unbounded, cancelling coefficients}}
$$

*Key takeaway:* The numerical statement (ill-conditioning inflates the solution along small-$\sigma$ directions) and the statistical statement (high variance under collinearity) are the same statement, and the regularizer is the same cure in both languages.

## Level 3 — Challenge

### Problem L3.1: Prove $\kappa_2(A^{\top}A) = \kappa_2(A)^2$, and the associated norm identities

Prove that for full-column-rank $A \in \mathbb{R}^{m \times n}$ the condition number squares, establishing along the way that $\Vert A \Vert_2 = \sigma_1$, $\Vert A^{+} \Vert_2 = 1/\sigma_n$, and $\Vert A^{\top}A \Vert_2 = \sigma_1^2$.

**Solution.**

**Step 1: $\Vert A \Vert_2 = \sigma_1$.** With the full SVD $A = U\Sigma V^{\top}$ and orthogonal $U, V$,

$$
\Vert A \Vert_2 = \max_{\Vert \mathbf{x} \Vert_2 = 1}\Vert U\Sigma V^{\top}\mathbf{x} \Vert_2 = \max_{\Vert \mathbf{y} \Vert_2 = 1}\Vert \Sigma\mathbf{y} \Vert_2 = \max_{\Vert \mathbf{y} \Vert_2 = 1}\sqrt{\sum_i \sigma_i^2 y_i^2} = \sigma_1 ,
$$

using that orthogonal maps preserve the 2-norm (so the substitution $\mathbf{y} = V^{\top}\mathbf{x}$ is a bijection of the unit sphere), and that the weighted sum is maximized by putting all mass on the largest $\sigma_i$.

**Step 2: $\Vert A^{+} \Vert_2 = 1/\sigma_n$.** $A^{+} = V\Sigma^{-1}U^{\top}$ (full column rank, so $\Sigma$ is invertible), and its singular values are $\{1/\sigma_i\}$; the largest of these is $1/\sigma_n$. Hence $\kappa_2(A) := \Vert A \Vert_2\Vert A^{+} \Vert_2 = \sigma_1/\sigma_n$, reconciling the two standard definitions.

**Step 3: The spectrum of $A^{\top}A$.**

$$
A^{\top}A = (U\Sigma V^{\top})^{\top}(U\Sigma V^{\top}) = V\Sigma^{\top}(U^{\top}U)\Sigma V^{\top} = V(\Sigma^{\top}\Sigma)V^{\top} = V\operatorname{diag}(\sigma_1^2, \ldots, \sigma_n^2)V^{\top},
$$

a symmetric eigendecomposition with orthogonal $V$. So the eigenvalues of $A^{\top}A$ are exactly $\sigma_i^2 \gt 0$, and since $A^{\top}A$ is symmetric its singular values coincide with its eigenvalues: $\Vert A^{\top}A \Vert_2 = \sigma_1^2$ and $\Vert (A^{\top}A)^{-1} \Vert_2 = 1/\sigma_n^2$.

**Step 4: Conclusion.**

$$
\kappa_2(A^{\top}A) = \Vert A^{\top}A \Vert_2 \, \Vert (A^{\top}A)^{-1} \Vert_2 = \frac{\sigma_1^2}{\sigma_n^2} = \kappa_2(A)^2 . \qquad \blacksquare
$$

**Sharpness.** The result is an identity, not a bound: nothing about $A$ can soften it, which is why the loss is *structural*. It also explains the empirical threshold $\kappa_2(A) \approx \varepsilon_{\text{mach}}^{-1/2} \approx 10^{8}$: beyond it, $\varepsilon_{\text{mach}}\kappa_2(A)^2 \gt 1$ and the normal equations cannot deliver a single correct digit.

$$
\boxed{\Vert A \Vert_2 = \sigma_1, \quad \Vert A^{+}\Vert_2 = \sigma_n^{-1}, \quad \kappa_2(A^{\top}A) = \kappa_2(A)^2}
$$

*Key takeaway:* Squaring a matrix squares the spread of its singular values, and the condition number is exactly that spread — the doubling of digit loss is an algebraic identity, not a pessimistic estimate.

### Problem L3.2: The Moore–Penrose conditions and the uniqueness of $A^{+}$

Verify that $A^{+} = V_r\Sigma_r^{-1}U_r^{\top}$ satisfies the four Penrose conditions, and prove that any matrix satisfying them is unique — hence that "the" pseudoinverse is well defined.

**Solution.**

**The four conditions.** $X = A^{+}$ must satisfy

$$
\text{(i) } AXA = A, \qquad \text{(ii) } XAX = X, \qquad \text{(iii) } (AX)^{\top} = AX, \qquad \text{(iv) } (XA)^{\top} = XA .
$$

**Verification.** Write $A = U_r\Sigma_r V_r^{\top}$, $X = V_r\Sigma_r^{-1}U_r^{\top}$, and use $U_r^{\top}U_r = V_r^{\top}V_r = I_r$. Then

$$
AX = U_r\Sigma_r V_r^{\top}V_r\Sigma_r^{-1}U_r^{\top} = U_rU_r^{\top}, \qquad XA = V_r V_r^{\top} .
$$

Both are symmetric (each is of the form $WW^{\top}$), giving (iii) and (iv); they are the orthogonal projectors onto $\mathcal{R}(A)$ and $\mathcal{R}(A^{\top})$ respectively. Then

$$
AXA = U_rU_r^{\top}\,U_r\Sigma_rV_r^{\top} = U_r\Sigma_rV_r^{\top} = A, \qquad XAX = V_rV_r^{\top}\,V_r\Sigma_r^{-1}U_r^{\top} = X,
$$

giving (i) and (ii).

**Uniqueness.** Let $X$ and $Y$ both satisfy (i)–(iv). First show $AX = AY$:

$$
AX \overset{\text{(i) for }Y}{=} (AYA)X = (AY)(AX) \overset{\text{(iii)}}{=} (AY)^{\top}(AX)^{\top} = Y^{\top}A^{\top}X^{\top}A^{\top} = Y^{\top}(AXA)^{\top} \overset{\text{(i) for }X}{=} Y^{\top}A^{\top} = (AY)^{\top} \overset{\text{(iii)}}{=} AY .
$$

The mirror argument with (iv) shows $XA = YA$:

$$
XA = X(AYA) = (XA)(YA) = (XA)^{\top}(YA)^{\top} = A^{\top}X^{\top}A^{\top}Y^{\top} = (AXA)^{\top}Y^{\top} = A^{\top}Y^{\top} = (YA)^{\top} = YA .
$$

Combining, using (ii) for each:

$$
X = XAX = X(AY) = (XA)Y = (YA)Y = YAY = Y . \qquad \blacksquare
$$

**Consequences.** Because $AA^{+} = U_rU_r^{\top} = P_{\mathcal{R}(A)}$, the least-squares fitted values are $AA^{+}\mathbf{b} = P\mathbf{b}$ — the pseudoinverse *is* the projection machinery of Proof 1. And when $A$ has full column rank, $A^{+} = (A^{\top}A)^{-1}A^{\top}$; when it has full row rank, $A^{+} = A^{\top}(AA^{\top})^{-1}$; when it is square and invertible, $A^{+} = A^{-1}$.

$$
\boxed{A^{+} \text{ is the unique matrix with } AXA = A,\ XAX = X,\ (AX)^{\top}=AX,\ (XA)^{\top}=XA}
$$

*Key takeaway:* The pseudoinverse is characterized purely algebraically — no SVD needed for the definition — and the two symmetry conditions are exactly what force $AA^{+}$ and $A^{+}A$ to be *orthogonal* (rather than oblique) projectors, which is what ties $A^{+}$ to the 2-norm.

### Problem L3.3: Sensitivity of the least-squares solution

Derive the first-order sensitivity of $\hat{\mathbf{x}}$ to a perturbation of $\mathbf{b}$, and explain the origin of the $\kappa_2(A)^2\tan\theta$ term appearing when $A$ is perturbed, where $\sin\theta = \rho/\Vert \mathbf{b}\Vert_2$.

**Solution.**

**Perturbing $\mathbf{b}$ only.** With $A$ full column rank, $\hat{\mathbf{x}} = A^{+}\mathbf{b}$ is *linear* in $\mathbf{b}$, so $\Delta\hat{\mathbf{x}} = A^{+}\Delta\mathbf{b}$ exactly and

$$
\Vert \Delta\hat{\mathbf{x}} \Vert_2 \le \Vert A^{+} \Vert_2 \Vert \Delta\mathbf{b} \Vert_2 = \frac{\Vert \Delta\mathbf{b}\Vert_2}{\sigma_n} .
$$

To make this relative, note $A^{+}$ annihilates the component of $\mathbf{b}$ orthogonal to $\mathcal{R}(A)$, so only $P\mathbf{b}$ drives the solution: $\Vert A\hat{\mathbf{x}} \Vert_2 = \Vert P\mathbf{b} \Vert_2 = \Vert \mathbf{b} \Vert_2\cos\theta$, where $\theta$ is the angle between $\mathbf{b}$ and $\mathcal{R}(A)$, i.e. $\sin\theta = \rho/\Vert \mathbf{b}\Vert_2$. Since $\Vert A\hat{\mathbf{x}} \Vert_2 \le \sigma_1\Vert \hat{\mathbf{x}} \Vert_2$,

$$
\frac{\Vert \Delta\hat{\mathbf{x}} \Vert_2}{\Vert \hat{\mathbf{x}} \Vert_2} \le \frac{\Vert \Delta\mathbf{b}\Vert_2/\sigma_n}{\Vert \mathbf{b}\Vert_2\cos\theta/\sigma_1} = \frac{\kappa_2(A)}{\cos\theta}\cdot\frac{\Vert \Delta\mathbf{b}\Vert_2}{\Vert \mathbf{b}\Vert_2} .
$$

So perturbing the right-hand side costs $\kappa_2(A)/\cos\theta$ — the condition number, inflated when $\mathbf{b}$ is nearly orthogonal to the column space (a fit that explains almost nothing).

**Perturbing $A$.** Differentiate $\hat{\mathbf{x}} = (A^{\top}A)^{-1}A^{\top}\mathbf{b}$ using $\delta(M^{-1}) = -M^{-1}(\delta M)M^{-1}$ with $M = A^{\top}A$:

$$
\Delta\hat{\mathbf{x}} = (A^{\top}A)^{-1}\Delta A^{\top}\underbrace{(\mathbf{b} - A\hat{\mathbf{x}})}_{\mathbf{r}} \;-\; A^{+}\Delta A\,\hat{\mathbf{x}} + O(\Vert \Delta A\Vert^2),
$$

after cancelling the terms that combine into $A^{+}$. **Two distinct mechanisms appear**, and this is the crux:

1. The term $-A^{+}\Delta A\hat{\mathbf{x}}$ involves $A^{+}$ once, contributing $\kappa_2(A)$.
2. The term $(A^{\top}A)^{-1}\Delta A^{\top}\mathbf{r}$ involves $(A^{\top}A)^{-1}$, whose norm is $1/\sigma_n^2$ — **two** factors of $\sigma_n^{-1}$, contributing $\kappa_2(A)^2$ — but weighted by the residual $\mathbf{r}$.

Normalizing the second term by $\Vert \hat{\mathbf{x}} \Vert_2 \ge \Vert \mathbf{b}\Vert_2\cos\theta/\sigma_1$ and writing $\Vert \mathbf{r}\Vert_2 = \Vert \mathbf{b}\Vert_2\sin\theta$ turns the ratio $\Vert \mathbf{r}\Vert_2/\Vert \hat{\mathbf{x}}\Vert_2$ into a factor $\sigma_1\tan\theta$, producing the classical bound

$$
\frac{\Vert \Delta\hat{\mathbf{x}}\Vert_2}{\Vert \hat{\mathbf{x}}\Vert_2} \lesssim \left(\kappa_2(A) + \kappa_2(A)^2\tan\theta\right)\frac{\Vert \Delta A\Vert_2}{\Vert A \Vert_2} .
$$

$$
\boxed{\text{perturbing } \mathbf{b}: \ \frac{\kappa_2(A)}{\cos\theta}; \qquad \text{perturbing } A: \ \kappa_2(A) + \kappa_2(A)^2\tan\theta, \quad \sin\theta = \rho/\Vert\mathbf{b}\Vert_2}
$$

*Key takeaway:* $\kappa_2(A)^2$ appears in the *problem's* conditioning too — but only multiplied by $\tan\theta$. For a good fit ($\theta \approx 0$) the problem is only $\kappa_2(A)$-conditioned, and a backward-stable QR attains that; the normal equations pay $\kappa_2(A)^2$ regardless of how good the fit is, which is precisely their unnecessary loss.

### Problem L3.4: Monotonicity of the ridge path and the limit $\lambda \to 0^{+}$

Prove that $\Vert \hat{\mathbf{x}}_\lambda \Vert_2$ is nonincreasing and $\Vert A\hat{\mathbf{x}}_\lambda - \mathbf{b} \Vert_2$ is nondecreasing in $\lambda \gt 0$, and that $\hat{\mathbf{x}}_\lambda \to A^{+}\mathbf{b}$ as $\lambda \to 0^{+}$ even when $A$ is rank deficient.

**Solution.**

Write $\phi(\mathbf{x}) = \Vert A\mathbf{x} - \mathbf{b}\Vert_2^2$ and $\psi(\mathbf{x}) = \Vert \mathbf{x}\Vert_2^2$, and abbreviate $\phi_\lambda = \phi(\hat{\mathbf{x}}_\lambda)$, $\psi_\lambda = \psi(\hat{\mathbf{x}}_\lambda)$.

**Step 1: monotonicity by an exchange argument (no calculus needed).** Let $0 \lt \lambda_1 \lt \lambda_2$. Optimality of each iterate for its own objective gives

$$
\phi_{\lambda_1} + \lambda_1\psi_{\lambda_1} \le \phi_{\lambda_2} + \lambda_1\psi_{\lambda_2}, \qquad \phi_{\lambda_2} + \lambda_2\psi_{\lambda_2} \le \phi_{\lambda_1} + \lambda_2\psi_{\lambda_1} .
$$

Adding the two inequalities, the $\phi$ terms cancel:

$$
\lambda_1\psi_{\lambda_1} + \lambda_2\psi_{\lambda_2} \le \lambda_1\psi_{\lambda_2} + \lambda_2\psi_{\lambda_1} \implies (\lambda_2 - \lambda_1)(\psi_{\lambda_2} - \psi_{\lambda_1}) \le 0 .
$$

Since $\lambda_2 \gt \lambda_1$, we get $\psi_{\lambda_2} \le \psi_{\lambda_1}$: **the solution norm is nonincreasing in $\lambda$.** Substituting back into the first inequality, $\phi_{\lambda_1} \le \phi_{\lambda_2} + \lambda_1(\psi_{\lambda_2} - \psi_{\lambda_1}) \le \phi_{\lambda_2}$: **the residual is nondecreasing in $\lambda$.** $\blacksquare$

**Step 2: the SVD confirms it quantitatively.** From Proof 6, $\hat{\mathbf{x}}_\lambda = \sum_{i \le r} \frac{\sigma_i(\mathbf{u}_i^{\top}\mathbf{b})}{\sigma_i^2 + \lambda}\mathbf{v}_i$, so by orthonormality of the $\mathbf{v}_i$,

$$
\Vert \hat{\mathbf{x}}_\lambda \Vert_2^2 = \sum_{i=1}^{r}\frac{\sigma_i^2 (\mathbf{u}_i^{\top}\mathbf{b})^2}{(\sigma_i^2 + \lambda)^2}, \qquad \Vert A\hat{\mathbf{x}}_\lambda - \mathbf{b} \Vert_2^2 = \sum_{i=1}^{r}\frac{\lambda^2 (\mathbf{u}_i^{\top}\mathbf{b})^2}{(\sigma_i^2 + \lambda)^2} + \sum_{i \gt r} (\mathbf{u}_i^{\top}\mathbf{b})^2 ,
$$

using $\sigma_i\cdot\frac{\sigma_i}{\sigma_i^2+\lambda} - 1 = \frac{-\lambda}{\sigma_i^2 + \lambda}$ for the second identity. Every term of the first sum decreases in $\lambda$ and every term of the second increases — the monotone trade-off, term by term. Plotting $\log\Vert \hat{\mathbf{x}}_\lambda\Vert_2$ against $\log\Vert A\hat{\mathbf{x}}_\lambda - \mathbf{b}\Vert_2$ traces Hansen's **L-curve**, whose corner is the standard heuristic for choosing $\lambda$.

**Step 3: the limit.** As $\lambda \to 0^{+}$, each coefficient $\frac{\sigma_i}{\sigma_i^2 + \lambda} \to \frac{1}{\sigma_i}$ for every $i \le r$ (the sum runs only over *nonzero* singular values, so no division by zero occurs). Hence

$$
\lim_{\lambda \to 0^{+}}\hat{\mathbf{x}}_\lambda = \sum_{i=1}^{r}\frac{\mathbf{u}_i^{\top}\mathbf{b}}{\sigma_i}\mathbf{v}_i = A^{+}\mathbf{b} ,
$$

which holds **regardless of rank**: the directions with $\sigma_i = 0$ never enter the expansion, so the limit automatically selects the minimum-norm solution. As $\lambda \to \infty$, every coefficient $\to 0$ and $\hat{\mathbf{x}}_\lambda \to \mathbf{0}$.

$$
\boxed{\lambda \mapsto \Vert \hat{\mathbf{x}}_\lambda \Vert_2 \text{ nonincreasing}, \quad \lambda \mapsto \rho(\lambda) \text{ nondecreasing}, \quad \hat{\mathbf{x}}_{0^{+}} = A^{+}\mathbf{b}, \quad \hat{\mathbf{x}}_{\infty} = \mathbf{0}}
$$

*Key takeaway:* The ridge path is a monotone one-parameter family sweeping from the minimum-norm least-squares solution to zero; the exchange argument proving it uses nothing but optimality of each point for its own objective, so it applies verbatim to lasso and to any convex penalty.